# Phase 2 — Data Cleaning: Verification & Audit Layer

**Status:** ✅ Completed (pipeline v2.0)  
**Authoritative Script:** `ml_service/src/clean.py`  
**Source Artifact:** `artifacts/cleaning_summary.json` & `artifacts/cleaned/*.csv`  

---

## Executive Summary
This notebook provides an interactive audit of the Phase 2 data cleaning stage. The pipeline processed **2,830,743 raw flows**, purged **2,867 rows with NaN/Inf rates** and **307,078 duplicate records**, outputting **2,520,798 clean unique flows** across 8 standardized CSVs in `artifacts/cleaned/`.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

with open(Paths.ARTIFACTS_DIR / 'cleaning_summary.json', 'r', encoding='utf-8') as f:
    cs = json.load(f)

print(f"Phase:            {cs['phase']}")
print(f"Pipeline Version: {cs['pipeline_version']}")
print(f"Elapsed Time:     {cs['elapsed_seconds']:.2f} s")
print(f"Peak Resident RAM:{cs['memory']['peak_rss_mb']:.2f} MB (Budget: {cs['memory']['memory_limit_mb']} MB)")

## 1. Flow Reduction Waterfall Breakdown
Accounting for every purged flow from raw ingress to clean storage:

In [ ]:
ga = cs['global_accounting']
d = ga['duplicate_rows_removed']

stages = ['Raw Ingress', 'NaN / Inf Purged', 'Within-File Dupes', 'Cross-File Dupes', 'Clean Output']
counts = [
    ga['rows_read'],
    ga['rows_with_nan_or_inf_removed'],
    d['within_file'],
    d['cross_file'],
    ga['rows_written']
]

df_waterfall = pd.DataFrame({'Pipeline Stage': stages, 'Flow Count': counts})

plt.figure(figsize=(10, 5))
sns.set_theme(style='whitegrid')
colors = ['#3498db', '#e74c3c', '#e67e22', '#f39c12', '#2ecc71']
barplot = sns.barplot(x='Pipeline Stage', y='Flow Count', data=df_waterfall, palette=colors, hue='Pipeline Stage', legend=False)
plt.yscale('log')
plt.title('Data Pipeline Flow Accounting (Log Scale)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Flow Count (Log Scale)', fontsize=12)

for p in barplot.patches:
    height = p.get_height()
    barplot.annotate(f'{int(height):,}',
                     (p.get_x() + p.get_width() / 2., height),
                     ha='center', va='bottom',
                     xytext=(0, 4), textcoords='offset points',
                     fontsize=10, fontweight='semibold')

plt.tight_layout()
plt.show()

## 2. Per-File Cleaning Breakdown
Tracking flow preservation and removal per capture day:

In [ ]:
records = []
for fr in cs['per_file_reports']:
    d = fr['duplicate_rows_removed']
    records.append({
        'File Name': fr['file_name'],
        'Rows Read': fr['rows_read'],
        'NaN/Inf Purged': fr['rows_with_nan_or_inf_removed'],
        'Within-File Dupes': d['within_file'],
        'Cross-File Dupes': d['cross_file'],
        'Clean Written': fr['rows_written'],
        'Retention %': f"{(fr['rows_written'] / fr['rows_read']) * 100:.2f}%"
    })

df_per_file = pd.DataFrame(records)
df_per_file.style.format({'Rows Read': '{:,}', 'Clean Written': '{:,}', 'NaN/Inf Purged': '{:,}', 'Within-File Dupes': '{:,}', 'Cross-File Dupes': '{:,}'})

## 3. Label Normalization & Encoding Fixes
Resolving legacy character encoding artifacts (e.g. `Web Attack � Brute Force` $\to$ `Web Attack - Brute Force`):

In [ ]:
norm_records = []
for raw_label, count in cs['label_normalization_audit']['raw_labels_detected'].items():
    normalized = raw_label.replace('\ufffd', '-').strip()
    norm_records.append({
        'Raw Label in CSV': repr(raw_label),
        'Clean Canonical Category': normalized,
        'Occurrences Corrected': count
    })

pd.DataFrame(norm_records)